In [85]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import albumentations as A
from PIL import Image
from tqdm import tqdm

In [4]:
img_paths = glob.glob("../data/raw/competitions/csiro-biomass/train/*")

In [37]:
aug_dir_path = Path('../data/augmented/image')

In [11]:
len(img_paths)

357

In [14]:
augmentation = [
    # ori aug
    A.HorizontalFlip(p=1),
    A.VerticalFlip(p=1),
    A.RandomBrightnessContrast(
        brightness_limit=0.25, 
        contrast_limit=0.25, 
        p=1
    ),
    A.HueSaturationValue(
        hue_shift_limit=15, 
        sat_shift_limit=25, 
        val_shift_limit=15, 
        p=1
    ),
    A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1),
    A.GaussNoise(std_range=[0.1, 0.2], mean_range=[0, 0], per_channel=True, noise_scale_factor=1, p=1.0),
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
    A.MotionBlur(blur_limit=5, p=1.0),
    A.GaussianBlur(blur_limit=5, p=1.0),
    A.RandomGamma(gamma_limit=(80, 120), p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.CoarseDropout(num_holes_range=[1, 10], hole_height_range=[10, 100], hole_width_range=[10, 100], fill=0, p=1),

    # horizontal flip aug
    A.Compose([A.HorizontalFlip(p=1), A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=1)]),
    A.Compose([A.HorizontalFlip(p=1), A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=1)]),
    A.Compose([A.HorizontalFlip(p=1), A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1)]),

    # vertical flip aug
    A.Compose([A.VerticalFlip(p=1), A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=1)]),
    A.Compose([A.VerticalFlip(p=1), A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=1)]),
    A.Compose([A.VerticalFlip(p=1), A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1)]),
]

In [16]:
total_aug_images = len(img_paths) * len(augmentation)
print(f"Total augmented images: {total_aug_images}")

Total augmented images: 6426


In [55]:
d_train = pd.read_csv("../data/raw/competitions/csiro-biomass/train.csv")

In [117]:
d_train_pivot = d_train.pivot(index='image_path', columns='target_name', values='target').reset_index()

In [118]:
d_train_pivot.columns.name = None

In [119]:
d_train_pivot['filename'] = d_train_pivot.image_path.apply(lambda x: x.split('/')[-1].replace('.jpg', ''))

In [90]:
data_aug = []
pbar = tqdm(total=d_train_pivot.shape[0] * len(augmentation))
for idx, row in d_train_pivot.iterrows():
    img = Image.open(os.path.join("../data/raw/competitions/csiro-biomass", row.image_path))
    img_arr = np.array(img)
    for aug_id, aug in enumerate(augmentation, 1):
        img_arr_aug = aug(image=img_arr)['image']
        img_aug = Image.fromarray(img_arr_aug)

        filename = Path(row.image_path).stem
        filename = f"{filename}_{aug_id}.jpg"
        save_img_path = (aug_dir_path / filename)
        img_aug.save(save_img_path)
        
        data = row.to_dict()
        data['image_path'] = os.path.join('augmented', 'image', filename)
        data_aug.append(data)
        pbar.update(1)

100%|███████████████████████████████████████████████████████████████████████████████| 6426/6426 [02:41<00:00, 47.30it/s]

In [91]:
d_data_aug = pd.DataFrame(data_aug)

In [92]:
d_data_aug.to_csv("../data/augmented/data.csv", index=False)

In [93]:
pd.read_csv("../data/augmented/data.csv")

,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,augmented/image/ID1011485656_1.jpg,0.0,31.9984,16.2751,48.2735,16.275
1,augmented/image/ID1011485656_2.jpg,0.0,31.9984,16.2751,48.2735,16.275
2,augmented/image/ID1011485656_3.jpg,0.0,31.9984,16.2751,48.2735,16.275
3,augmented/image/ID1011485656_4.jpg,0.0,31.9984,16.2751,48.2735,16.275
4,augmented/image/ID1011485656_5.jpg,0.0,31.9984,16.2751,48.2735,16.275
...,...,...,...,...,...,...
6421,augmented/image/ID983582017_14.jpg,0.0,0.0000,40.9400,40.9400,40.940
6422,augmented/image/ID983582017_15.jpg,0.0,0.0000,40.9400,40.9400,40.940
6423,augmented/image/ID983582017_16.jpg,0.0,0.0000,40.9400,40.9400,40.940
6424,augmented/image/ID983582017_17.jpg,0.0,0.0000,40.9400,40.9400,40.940


In [95]:
d_data_head = d_data_aug.head().copy()

In [96]:
d_data_tail = d_data_aug.tail().copy()

In [103]:
d_concat = pd.concat((d_data_head, d_data_tail), axis=0)

In [110]:
d_concat['filename'] = d_concat.image_path.apply(lambda x: x.split('/')[-1].split('_')[0])

In [121]:
d_concat[d_concat.filename.isin(d_train_pivot.filename)]

,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,filename
0,augmented/image/ID1011485656_1.jpg,0.0,31.9984,16.2751,48.2735,16.275,ID1011485656
1,augmented/image/ID1011485656_2.jpg,0.0,31.9984,16.2751,48.2735,16.275,ID1011485656
2,augmented/image/ID1011485656_3.jpg,0.0,31.9984,16.2751,48.2735,16.275,ID1011485656
3,augmented/image/ID1011485656_4.jpg,0.0,31.9984,16.2751,48.2735,16.275,ID1011485656
4,augmented/image/ID1011485656_5.jpg,0.0,31.9984,16.2751,48.2735,16.275,ID1011485656
6421,augmented/image/ID983582017_14.jpg,0.0,0.0000,40.9400,40.9400,40.940,ID983582017
6422,augmented/image/ID983582017_15.jpg,0.0,0.0000,40.9400,40.9400,40.940,ID983582017
6423,augmented/image/ID983582017_16.jpg,0.0,0.0000,40.9400,40.9400,40.940,ID983582017
6424,augmented/image/ID983582017_17.jpg,0.0,0.0000,40.9400,40.9400,40.940,ID983582017
6425,augmented/image/ID983582017_18.jpg,0.0,0.0000,40.9400,40.9400,40.940,ID983582017


In [109]:
d_train_pivot

,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,filaname
0,train/ID1011485656.jpg,0.0000,31.9984,16.2751,48.2735,16.2750,ID1011485656
1,train/ID1012260530.jpg,0.0000,0.0000,7.6000,7.6000,7.6000,ID1012260530
2,train/ID1025234388.jpg,6.0500,0.0000,0.0000,6.0500,6.0500,ID1025234388
3,train/ID1028611175.jpg,0.0000,30.9703,24.2376,55.2079,24.2376,ID1028611175
4,train/ID1035947949.jpg,0.4343,23.2239,10.5261,34.1844,10.9605,ID1035947949
...,...,...,...,...,...,...,...
352,train/ID975115267.jpg,40.0300,0.0000,0.8000,40.8300,40.8300,ID975115267
353,train/ID978026131.jpg,24.6445,4.1948,12.0601,40.8994,36.7046,ID978026131
354,train/ID980538882.jpg,0.0000,1.1457,91.6543,92.8000,91.6543,ID980538882
355,train/ID980878870.jpg,32.3575,0.0000,2.0325,34.3900,34.3900,ID980878870
